# RAG Demo (Colab)
Simple, minimal notebook to run the RAG system.

**No HuggingFace token required** - uses ungated Mistral-7B model.
Just upload PDFs and run all cells.

## Setup Instructions

**Simple setup - no token, no Drive downloads needed:**
1. Clone repo (automatic)
2. Install dependencies (automatic)
3. Upload 2 PDFs when prompted
4. Model downloads automatically from HuggingFace (~14GB, first run only)

In [ ]:
# Install dependencies
%pip install -q -r /content/naive_rag/requirements.txt

In [ ]:
# Clone repo (skip if already in /content/naive_rag)
import os, subprocess, textwrap
repo_dir = "/content/naive_rag"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/arun41687/naive_rag_hf.git", repo_dir], check=True)
os.chdir(repo_dir)
print("Repo ready:", os.getcwd())

In [ ]:
# Upload PDFs (required - repo doesn't include them)
import os
try:
    from google.colab import files
    print("📤 Please upload the 2 PDF files:")
    uploaded = files.upload()
    for name in uploaded.keys():
        print(f"✅ Uploaded: {name}")
except Exception as exc:
    print("⚠️  Upload skipped (not in Colab):", exc)

print("\nCurrent dir:", os.getcwd())
print("Available PDFs:", [f for f in os.listdir('.') if f.lower().endswith('.pdf')])

In [ ]:
# Index documents and run a sample query
from rag_system import RAGSystem

documents = [
    {"path": "10-Q4-2024-As-Filed.pdf", "name": "Apple 10-K"},
    {"path": "tsla-20231231-gen.pdf", "name": "Tesla 10-K"},
]

print("🚀 Initializing RAG system...")
print("   Model: mistralai/Mistral-7B-Instruct-v0.2 (ungated, no token needed)")
print("   First run will download ~14GB (cached for future runs)")

rag = RAGSystem(
    model_name="mistralai/Mistral-7B-Instruct-v0.2",
    embedding_model="all-MiniLM-L6-v2",
    use_reranker=True
)

print("\n📊 Ingesting documents...")
rag.ingest_documents(documents)
rag.save_index("./rag_index")

print("\n❓ Running sample query...")
result = rag.answer_question("What was Apple's total revenue for 2024?")
print(f"\n✅ Answer: {result['answer']}")
print(f"📚 Sources: {result['sources']}")

In [ ]:
# Optional: run full evaluation
from rag_system import run_evaluation
run_evaluation(rag)